# Winner-set stability

For each sampled group, this notebook measures the stability of complete terminal-payoff winner sets across episodes using Nogueira's chance-corrected stability measure. A value of 1 indicates that every episode has the same winner set, 0 indicates recurrence expected under random winner identities, and negative values indicate less recurrence than expected by chance. One stability value is computed per simulated group.

In [ ]:
# Input: completed policy-run store.
RUN_PATH = "artifacts/simulation_runs/run-20260729T011128.zarr"

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "pyproject.toml").exists() and (parent / "tasks").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
RUN_PATH = Path(RUN_PATH).expanduser()
if not RUN_PATH.is_absolute():
    RUN_PATH = PROJECT_ROOT / RUN_PATH
RUN_PATH = RUN_PATH.resolve()

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scienceplots  # noqa: F401 - registers the science style
import xarray as xr
from IPython.display import display
from matplotlib.colors import Normalize
from matplotlib_inline.backend_inline import set_matplotlib_formats
from scipy.stats import bootstrap, friedmanchisquare

from network_creation.stats.topology_welfare import (
    reference_centered_mean_welfare,
)
from tasks.policies.outputs import SAMPLED_AGENT_PARAMETERS_FIELD
from tasks.policies.results import load_policy_results

%matplotlib inline
plt.style.use("science")
set_matplotlib_formats("svg")
plt.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 17,
    "axes.labelsize": 15,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
    "legend.title_fontsize": 13,
    "figure.titlesize": 19,
})

PAYOFF_NAME = "agent_payoff.runs"
TIE_ATOL = 1e-7
RNG_SEED = 20260729
BOOTSTRAP_RESAMPLES = 5_000
CONFIDENCE_LEVEL = 0.95
FILTER_ABOVE_MAX_RANDOM_WELFARE = False
WELFARE_SELECTION_LABEL = (
    "Welfare above Max ER"
    if FILTER_ABOVE_MAX_RANDOM_WELFARE
    else "All groups"
)

## Load and validate the artifact

`load_policy_results` reconstructs the run as labeled arrays. The cell below reduces its singleton policy-composition and normalized terminal-time axes, then creates one `xarray.DataTree` whose `"10"`, `"50"`, and `"100"` children hold exact-shape group-size data. Each size node stores `(group, agent, episode)` payoffs, episode-averaged normalized group welfare, its normalized Max ER reference, and named group and individual parameters.

In [ ]:
loaded = load_policy_results(
    RUN_PATH,
    stats=PAYOFF_NAME,
    load_sampled_agent_parameters=True,
    normalize_time=True,
)
if loaded.attrs["has_failures"]:
    raise RuntimeError("The analysis rejects stores containing failed rows")
if loaded.attrs["sweep_layout"] != "cartesian":
    raise ValueError("Expected a Cartesian sweep")

policy_proportion = float(loaded.best_response_proportion.item())
terminal_activation_time = float(loaded.step.item())
loaded = loaded.squeeze(
    dim=("best_response_proportion", "step"), drop=True
)
group_parameter_names = {
    "benefit_to_cost_ratio": "benefit_to_cost_ratio",
    "distance_discount": "distance_discount",
    "visibility_distance": "visibility_center",
    "outside_optimism": "outside_optimism_center",
    "exploration": "exploration_center",
    "agent.activation_weight": "activation_weight",
}


def build_size_nodes(
    results,
    n_agents,
    *,
    group_parameter_names,
    payoff_name,
    sampled_agent_parameters_field,
):
    source = results.sel(n_agents=n_agents)
    individual_parameters = {}
    for name in source.agent_parameter.values:
        name = str(name)
        individual_parameters[name] = (
            ("group", "agent"),
            source[sampled_agent_parameters_field]
            .sel(agent_parameter=name)
            .isel(agent_index=slice(0, n_agents))
            .values,
        )
    group_parameters = {
        variable_name: (
            ("group",), source.theta.sel(theta_parameter=source_name).values
        )
        for source_name, variable_name in group_parameter_names.items()
    }

    root = xr.Dataset(
        data_vars={
            "payoff": (
                ("group", "agent", "episode"),
                source[payoff_name]
                .isel(agent_slot=slice(0, n_agents))
                .transpose("parameter_row", "agent_slot", "episode")
                .values,
            ),
        },
        coords={
            "group": np.asarray(source.input_index.values, dtype=np.int64),
            "parameter_row": (
                "group", np.asarray(source.parameter_row.values, dtype=np.int64)
            ),
            "agent": np.arange(n_agents, dtype=np.int64),
        },
        attrs={"n_agents": n_agents},
    )
    size = str(n_agents)
    return {
        f"/{size}": root,
        f"/{size}/group_parameters": xr.Dataset(group_parameters),
        f"/{size}/individual_parameters": xr.Dataset(individual_parameters),
    }


n_agents_values = np.asarray(loaded.n_agents.values, dtype=np.int64)
tree_nodes = {
    "/": xr.Dataset(
        attrs={
            **loaded.attrs,
            "policy_proportion": policy_proportion,
            "terminal_activation_time": terminal_activation_time,
        },
    )
}
for n_agents in n_agents_values:
    tree_nodes.update(build_size_nodes(
        loaded,
        int(n_agents),
        group_parameter_names=group_parameter_names,
        payoff_name=PAYOFF_NAME,
        sampled_agent_parameters_field=SAMPLED_AGENT_PARAMETERS_FIELD,
    ))
analysis = xr.DataTree.from_dict(tree_nodes)
for n_agents in n_agents_values:
    size_data = analysis[str(n_agents)]
    group_parameters = size_data["group_parameters"]
    benefit_to_cost_ratio = group_parameters["benefit_to_cost_ratio"]
    distance_discount = group_parameters["distance_discount"]
    maximum_welfare = xr.DataArray(
        [
            reference_centered_mean_welfare(
                reference="max",
                n_agents=int(n_agents),
                benefit_to_cost_ratio=float(benefit),
                distance_discount=float(discount),
            )
            for benefit, discount in zip(
                benefit_to_cost_ratio.values,
                distance_discount.values,
                strict=True,
            )
        ],
        dims="group",
        coords={"group": size_data["group"]},
    )
    random_welfare = xr.DataArray(
        [
            reference_centered_mean_welfare(
                reference="random",
                n_agents=int(n_agents),
                benefit_to_cost_ratio=float(benefit),
                distance_discount=float(discount),
            )
            for benefit, discount in zip(
                benefit_to_cost_ratio.values,
                distance_discount.values,
                strict=True,
            )
        ],
        dims="group",
        coords={"group": size_data["group"]},
    )
    payoff_scale = (
        benefit_to_cost_ratio
        * (int(n_agents) - 1)
        * distance_discount
    )
    size_data["normalized_group_welfare"] = (
        size_data["payoff"].astype(np.float64).mean("agent")
        * payoff_scale
        / (maximum_welfare + 1e-8)
    ).mean("episode")
    size_data["normalized_max_random_welfare"] = (
        random_welfare / maximum_welfare
    )
del tree_nodes, loaded


n_episodes = analysis[str(n_agents_values[0])].dataset.sizes["episode"]
analysis

## Winner-set stability

For each episode, the winner set contains every player whose terminal payoff is within tolerance $\varepsilon$ of the maximum payoff. For one group, the resulting binary matrix records whether each player belongs to the winner set in each episode. Nogueira stability compares the observed variability of these membership indicators with the variability expected under random assignment of winner identities. It equals 1 when all episodes have the same winner set, 0 at chance, and can be negative when winner identities recur less often than expected by chance.

When `FILTER_ABOVE_MAX_RANDOM_WELFARE` is enabled, the analysis retains groups whose episode-averaged normalized welfare exceeds the maximum expected welfare of an Erdős-Rényi network, shown as `Max ER` in `task_result_loader.ipynb`. Otherwise, it uses all groups. Intervals obtained by resampling parameter rows describe sensitivity conditional on the current scrambled-Sobol design and simulator seed; they are not calibrated IID population confidence intervals.

In [ ]:
def nogueira_winner_set_stability(payoffs, epsilon=TIE_ATOL):
    """Return one Nogueira winner-set stability value per group."""
    payoffs = np.asarray(payoffs, dtype=np.float64)
    n_groups, _, n_episodes = payoffs.shape
    winner_sets = (
        payoffs
        >= payoffs.max(axis=1, keepdims=True) - epsilon
    )
    selection_frequency = winner_sets.mean(
        axis=2, dtype=np.float64
    )
    observed_variability = (
        n_episodes
        / (n_episodes - 1)
        * selection_frequency
        * (1.0 - selection_frequency)
    ).mean(axis=1)
    selected_fraction = winner_sets.mean(
        axis=(1, 2), dtype=np.float64
    )
    chance_variability = selected_fraction * (1.0 - selected_fraction)

    stability = np.empty(n_groups, dtype=np.float64)
    defined = chance_variability > 0.0
    stability[defined] = (
        1.0
        - observed_variability[defined]
        / chance_variability[defined]
    )
    stability[~defined] = 1.0
    return stability


def filter_above_max_random_welfare(data: xr.Dataset) -> xr.Dataset:
    """Return xarray rows whose normalized welfare exceeds Max ER."""
    return data.where(
        data["normalized_group_welfare"]
        > data["normalized_max_random_welfare"],
        drop=True,
    )


def bootstrap_row_interval(
    values, *, rng, n_resamples=5_000, confidence_level=0.95
):
    """Return the mean and a conditional row-bootstrap interval."""
    values = np.asarray(values, dtype=np.float64)
    if values.size < 2:
        mean = float(values.mean()) if values.size else np.nan
        return mean, np.nan, np.nan
    interval = bootstrap(
        (values,),
        np.mean,
        n_resamples=n_resamples,
        confidence_level=confidence_level,
        method="percentile",
        batch=64,
        rng=rng,
    ).confidence_interval
    return (
        float(values.mean()),
        float(interval.low),
        float(interval.high),
    )

In [ ]:
def plot_parameter_space(
    analysis, values_by_size, *, norm, colorbar_label, title
):
    """Plot one group-level statistic across sampled parameter centers."""
    n_agent_sizes = list(values_by_size)
    visibility_values = np.unique(
        analysis[str(n_agent_sizes[0])]
        ["group_parameters"]
        ["visibility_center"]
        .values
    )
    figure, axes = plt.subplots(
        len(n_agent_sizes),
        len(visibility_values),
        figsize=(3.1 * len(visibility_values), 2.8 * len(n_agent_sizes)),
        sharex=True,
        sharey=True,
        gridspec_kw={"wspace": 0.06, "hspace": 0.06},
        squeeze=False,
    )

    mapped = None
    for row_index, n_agents in enumerate(n_agent_sizes):
        size_data = analysis[str(n_agents)]
        values = values_by_size[n_agents]
        parameters = (
            size_data["group_parameters"].dataset
            .assign_coords(parameter_row=size_data["parameter_row"])
            .swap_dims({"group": "parameter_row"})
            .sel(parameter_row=values["parameter_row"])
        )
        for column_index, visibility in enumerate(visibility_values):
            axis = axes[row_index, column_index]
            mask = np.isclose(
                parameters["visibility_center"].values, visibility
            )
            mapped = axis.scatter(
                parameters["outside_optimism_center"].values[mask],
                parameters["exploration_center"].values[mask],
                c=values.values[mask],
                cmap="jet",
                norm=norm,
                s=7,
                linewidths=0,
                rasterized=True,
            )
            axis.set_box_aspect(1)
            axis.grid(False)
            axis.tick_params(axis="both", which="both", length=0)
            if row_index == 0:
                axis.set_title(f"Visibility center {visibility:g}")
            if column_index == 0:
                axis.set_ylabel(f"$n={n_agents}$\nExploration center")
            if row_index == len(n_agent_sizes) - 1:
                axis.set_xlabel("Outside-optimism center")

    colorbar = figure.colorbar(
        mapped, ax=axes.ravel(), fraction=0.025, pad=0.035, extend="max"
    )
    colorbar.set_label(colorbar_label)
    figure.suptitle(title)
    return figure

In [ ]:
stability_by_size = {}
for n_agents in sorted(n_agents_values):
    size_data = analysis[str(n_agents)]
    selected = size_data.dataset
    if FILTER_ABOVE_MAX_RANDOM_WELFARE:
        selected = filter_above_max_random_welfare(selected)
    payoffs = selected["payoff"].values
    observed = xr.DataArray(
        nogueira_winner_set_stability(payoffs),
        dims="parameter_row",
        coords={"parameter_row": selected["parameter_row"].values},
    )
    stability_by_size[int(n_agents)] = observed

In [ ]:
summary_rng = np.random.default_rng(RNG_SEED + 1)
summary_rows = []
for n_agents, observed in stability_by_size.items():
    mean, interval_low, interval_high = bootstrap_row_interval(
        observed,
        rng=summary_rng,
        n_resamples=BOOTSTRAP_RESAMPLES,
        confidence_level=CONFIDENCE_LEVEL,
    )
    summary_rows.append({
        "n_agents": n_agents,
        "groups": observed.size,
        "mean_nogueira_stability": mean,
        "row_interval_low": interval_low,
        "row_interval_high": interval_high,
        "median_nogueira_stability": (
            np.median(observed) if observed.size else np.nan
        ),
    })

summary = pd.DataFrame(summary_rows)
display(summary.style.format({
    "mean_nogueira_stability": "{:.4f}",
    "row_interval_low": "{:.4f}",
    "row_interval_high": "{:.4f}",
    "median_nogueira_stability": "{:.4f}",
}))

In [ ]:
lower_bound = -1.0 / (n_episodes - 1)
size_colors = dict(zip(
    stability_by_size,
    plt.rcParams["axes.prop_cycle"].by_key()["color"],
))

figure, axis = plt.subplots(figsize=(7, 7), layout="compressed")
for n_agents, observed in stability_by_size.items():
    if observed.size:
        axis.ecdf(
            observed.values,
            compress=True,
            color=size_colors[n_agents],
            linewidth=2.2,
            label=rf"$n={n_agents}$",
        )

axis.axvline(
    0.0, color="black", linestyle="--", linewidth=1.0
)
axis.set(
    title=f"Chance-corrected winner-set stability\n{WELFARE_SELECTION_LABEL}",
    xlabel=r"Nogueira winner-set stability $\Phi$",
    ylabel="Cumulative proportion of groups",
    xlim=(lower_bound, 1.0),
    ylim=(0.0, 1.0),
)
axis.set_box_aspect(1)
axis.grid(alpha=0.25)
axis.legend(title="Group size", frameon=False)
plt.show()

In [ ]:
stability_values = np.concatenate([
    values.values for values in stability_by_size.values()
])
stability_norm = Normalize(
    vmin=-1.0 / (n_episodes - 1),
    vmax=float(np.quantile(stability_values, 0.99)),
)
figure = plot_parameter_space(
    analysis,
    stability_by_size,
    norm=stability_norm,
    colorbar_label="Nogueira winner-set stability",
    title=(
        "Winner-set stability across parameter space\n"
        f"{WELFARE_SELECTION_LABEL}"
    ),
)
plt.show()

## Kendall's W rank concordance

Kendall's W measures whether the same agents retain similar terminal-payoff ranks across episodes. It ranges from 0 for no rank concordance to 1 for identical rankings. This section uses the same welfare-filter setting as the winner-set analysis. Shading shows a joint 95% simultaneous confidence band obtained by resampling aligned parameter rows.

In [ ]:
def kendall_concordance(payoffs, batch_size=512):
    """Return Kendall's W for every group."""
    payoffs = np.asarray(payoffs, dtype=np.float64)
    n_groups, n_agents, n_episodes = payoffs.shape
    result = np.full(n_groups, np.nan, dtype=np.float64)
    denominator = n_episodes * (n_agents - 1)
    for start in range(0, n_groups, batch_size):
        stop = min(start + batch_size, n_groups)
        batch = payoffs[start:stop]
        all_tied = np.all(batch == batch[:, :1, :], axis=(1, 2))
        valid = ~all_tied
        batch_result = np.full(stop - start, np.nan, dtype=np.float64)
        if np.any(valid):
            statistic = friedmanchisquare(
                *np.moveaxis(batch[valid], 1, 0), axis=-1
            ).statistic
            batch_result[valid] = np.asarray(statistic) / denominator
        result[start:stop] = batch_result
    return np.clip(result, 0.0, 1.0)


def joint_ecdf_band_half_width(
    values_by_size, rng, n_resamples, confidence
):
    """Return a joint bootstrap ECDF-band half-width."""
    n_rows = values_by_size.sizes["parameter_row"]
    encoded = {}
    observed_cdfs = {}
    for name, values in values_by_size.data_vars.items():
        values = values.values
        finite = np.isfinite(values)
        _, inverse, counts = np.unique(
            values[finite], return_inverse=True, return_counts=True
        )
        bins = np.full(n_rows, -1, dtype=np.int32)
        bins[finite] = inverse
        encoded[name] = bins
        observed_cdfs[name] = np.cumsum(counts) / counts.sum()

    maximum_deviations = np.empty(n_resamples, dtype=np.float64)
    for draw in range(n_resamples):
        indices = rng.integers(n_rows, size=n_rows)
        maximum_deviation = 0.0
        for name, bins in encoded.items():
            selected = bins[indices]
            selected = selected[selected >= 0]
            bootstrap_cdf = np.cumsum(np.bincount(
                selected, minlength=len(observed_cdfs[name])
            )) / len(selected)
            maximum_deviation = max(
                maximum_deviation,
                np.max(np.abs(bootstrap_cdf - observed_cdfs[name])),
            )
        maximum_deviations[draw] = maximum_deviation

    return float(np.quantile(maximum_deviations, confidence))

In [ ]:
kendall_by_size = {}
for n_agents in sorted(n_agents_values):
    size_data = analysis[str(n_agents)]
    selected = size_data.dataset
    if FILTER_ABOVE_MAX_RANDOM_WELFARE:
        selected = filter_above_max_random_welfare(selected)
    kendall_w = xr.DataArray(
        kendall_concordance(selected["payoff"].values),
        dims="parameter_row",
        coords={"parameter_row": selected["parameter_row"].values},
    ).dropna("parameter_row")
    kendall_by_size[int(n_agents)] = kendall_w

kendall_values = xr.Dataset({
    str(n_agents): values
    for n_agents, values in kendall_by_size.items()
}).reindex(
    parameter_row=analysis[str(n_agents_values[0])]["parameter_row"].values
)
kendall_band_half_width = joint_ecdf_band_half_width(
    kendall_values,
    np.random.default_rng(RNG_SEED + 3),
    BOOTSTRAP_RESAMPLES,
    CONFIDENCE_LEVEL,
)

In [ ]:
size_colors = dict(zip(
    kendall_by_size,
    plt.rcParams["axes.prop_cycle"].by_key()["color"],
))
figure, axis = plt.subplots(figsize=(7, 7), layout="compressed")
for n_agents, values in kendall_by_size.items():
    observed = values.values
    cdf_grid = np.unique(np.concatenate(([0.0], observed, [1.0])))
    observed_cdf = np.searchsorted(
        np.sort(observed), cdf_grid, side="right"
    ) / observed.size
    axis.fill_between(
        cdf_grid,
        np.clip(observed_cdf - kendall_band_half_width, 0.0, 1.0),
        np.clip(observed_cdf + kendall_band_half_width, 0.0, 1.0),
        step="post",
        color=size_colors[n_agents],
        alpha=0.12,
    )
    axis.ecdf(
        observed,
        compress=True,
        color=size_colors[n_agents],
        linewidth=2.2,
        label=rf"$n={n_agents}$",
    )

axis.set(
    title=f"Identity-linked rank persistence\n{WELFARE_SELECTION_LABEL}",
    xlabel="Kendall's W",
    ylabel="Cumulative proportion of groups",
    xlim=(0.0, 1.0),
    ylim=(0.0, 1.0),
)
axis.set_box_aspect(1)
axis.grid(alpha=0.25)
axis.legend(title="Group size", frameon=False)
plt.show()

In [ ]:
kendall_plot_values = np.concatenate([
    values.values for values in kendall_by_size.values()
])
kendall_norm = Normalize(
    vmin=0.0,
    vmax=float(np.quantile(kendall_plot_values, 0.99)),
)
figure = plot_parameter_space(
    analysis,
    kendall_by_size,
    norm=kendall_norm,
    colorbar_label="Kendall's W",
    title=(
        "Kendall's W across parameter space\n"
        f"{WELFARE_SELECTION_LABEL}"
    ),
)
plt.show()